# 💓 Module 5: ECG Arrhythmia Classification
**Dataset:** MIT-BIH Arrhythmia (PhysioNet) via Kaggle

**Classes:** Normal | Supraventricular | Ventricular | Fusion | Unclassifiable (5 classes)

**Model:** 1D-CNN + Bidirectional LSTM

**Output:** `ecg_model.h5` → place in `ai-server/models/`

---

In [ ]:
!pip install -q kaggle tensorflow numpy pandas matplotlib seaborn scikit-learn imbalanced-learn

In [ ]:
# ─── STEP 2: Download Dataset ─────────────────────────────────────────────────
# MIT-BIH preprocessed to 187 time-steps per beat
!kaggle datasets download -d shayanfazeli/heartbeat -p ./data --unzip
!ls ./data/

In [ ]:
# ─── STEP 3: Load Data ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('./data/mitbih_train.csv', header=None)
test_df = pd.read_csv('./data/mitbih_test.csv', header=None)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

# Last column is label
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values.astype(int)
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values.astype(int)

print('\nClass distribution (train):')
CLASS_NAMES = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']
for i, name in enumerate(CLASS_NAMES):
    count = (y_train == i).sum()
    print(f'  Class {i} ({name}): {count} ({count/len(y_train)*100:.1f}%)')

In [ ]:
# ─── STEP 4: EDA - Visualize ECG Beats ────────────────────────────────────────
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(14, 12))

for i, (cls_name, cls_idx) in enumerate(zip(CLASS_NAMES, range(5))):
    cls_samples = X_train[y_train == cls_idx][:3]
    for j, sample in enumerate(cls_samples):
        axes[i][j].plot(sample, color=['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6'][i], linewidth=1)
        axes[i][j].set_title(f'{cls_name}' if j==1 else '', fontsize=10)
        axes[i][j].set_ylim(-0.1, 1.1)
        axes[i][j].axis('off')
        if j == 0:
            axes[i][j].set_ylabel(cls_name, fontsize=9, fontweight='bold')
            axes[i][j].axis('on')
            axes[i][j].get_xaxis().set_visible(False)

plt.suptitle('ECG Beat Samples by Arrhythmia Class', fontsize=14)
plt.tight_layout()
plt.savefig('./data/ecg_samples.png', dpi=100)
plt.show()

In [ ]:
# ─── STEP 5: Handle Class Imbalance with Oversampling ────────────────────────
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print('After oversampling:')
for i, name in enumerate(CLASS_NAMES):
    count = (y_train_res == i).sum()
    print(f'  {name}: {count}')

# Reshape for 1D-CNN: (samples, timesteps, channels)
X_train_res = X_train_res.reshape(-1, 187, 1)
X_test_r = X_test.reshape(-1, 187, 1)
print('\nReshaped X_train:', X_train_res.shape)
print('Reshaped X_test:', X_test_r.shape)

In [ ]:
# ─── STEP 6: Build 1D CNN + BiLSTM Model ──────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def build_ecg_model(input_shape=(187, 1), num_classes=5):
    inputs = tf.keras.Input(shape=input_shape)
    
    # ── Block 1: Multi-scale CNN ──
    x1 = layers.Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x2 = layers.Conv1D(64, 5, activation='relu', padding='same')(inputs)
    x3 = layers.Conv1D(64, 7, activation='relu', padding='same')(inputs)
    x = layers.Concatenate()([x1, x2, x3])
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    
    # ── Block 2: Deeper CNN ──
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.25)(x)
    
    # ── Block 3: BiLSTM ──
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dropout(0.3)(x)
    
    # ── Classifier ──
    x = layers.Dense(128, activation='relu', 
                     kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return tf.keras.Model(inputs, outputs)

model = build_ecg_model()
model.summary()

In [ ]:
# ─── STEP 7: Train Model ──────────────────────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import os

os.makedirs('./output', exist_ok=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('./output/ecg_best.h5', save_best_only=True, monitor='val_accuracy')
]

history = model.fit(
    X_train_res, y_train_res,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks
)

In [ ]:
# ─── STEP 8: Evaluate ─────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(X_test_r), axis=1)

print('=== TEST RESULTS ===')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues')
axes[0].set_title('Confusion Matrix')
axes[0].tick_params(axis='x', rotation=45)

# Training curve
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('./data/ecg_results.png', dpi=100)
plt.show()

In [ ]:
# ─── STEP 9: Save Model ───────────────────────────────────────────────────────
model.save('./output/ecg_model.h5')

import json
meta = {
    'input_shape': [187, 1],
    'num_classes': 5,
    'class_names': CLASS_NAMES,
    'class_map': {str(i): name for i, name in enumerate(CLASS_NAMES)}
}
with open('./output/ecg_meta.json', 'w') as f:
    json.dump(meta, f)

print('✅ Saved: ./output/ecg_model.h5')
print('✅ Saved: ./output/ecg_meta.json')
print('\n📁 COPY BOTH FILES TO: healthvision-ai/ai-server/models/')

In [ ]:
# ─── STEP 10: Test Prediction (matches ecg_service.py format) ─────────────────
from tensorflow.keras.models import load_model
import json

loaded = load_model('./output/ecg_model.h5')
with open('./output/ecg_meta.json') as f:
    meta = json.load(f)

# Simulate a single ECG beat (187 samples)
sample_beat = X_test[0].reshape(1, 187, 1)
true_label = y_test[0]

probs = loaded.predict(sample_beat)[0]
pred_class = np.argmax(probs)

print('ECG Beat Classification:')
for i, (name, prob) in enumerate(zip(CLASS_NAMES, probs)):
    bar = '█' * int(prob * 30)
    marker = ' ← PREDICTED' if i == pred_class else ''
    print(f'  {name:<20} {bar:<30} {prob:.4f}{marker}')

print(f'\nTrue Label:  {CLASS_NAMES[true_label]}')
print(f'Predicted:   {CLASS_NAMES[pred_class]}')
print(f'Correct:     {"✅" if true_label == pred_class else "❌"}')